###Initialization of libraries and path

In [0]:
%sql
--Databricks RunTime and Spark Version
SELECT current_version().dbr_version;
SELECT version();


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

from datetime import datetime

CATALOG = "quickcart"

SOURCE_VOLUME = f"/Volumes/{CATALOG}/default/source_data"

BRONZE_SCHEMA = "bronze"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS quickcart.bronze.bronze_ingestion_audit(
    source_table STRING,
    target_table STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_seconds DOUBLE,
    status STRING,
    error_message STRING,
    record_count BIGINT
) USING DELTA;

In [0]:
%sql
SELECT *
FROM quickcart.bronze.bronze_ingestion_audit;

###Building the function for ingestion

In [0]:
def ingest_to_bronze(source_table, source_format = 'parquet'):
    source_path = f"{SOURCE_VOLUME}/{source_table}"
    schema_path = f"{SOURCE_VOLUME}/_schemas/{source_table}"
    checkpoint_path = f"{SOURCE_VOLUME}/_checkpoints/{source_table}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{source_table}"

    start_time = datetime.now()

    try:
        previous_count = spark.sql(
        f'''
        SELECT COUNT(*) AS count
        FROM {target_table}
        '''
        ).collect()[0]["count"]
    except Exception:
        previous_count = 0

    print("=" * 70)
    print(f"Starting Bronze ingestion: {source_table}")
    print("=" * 70)
    
    print(f"Source Path     : {source_path}")
    print(f"Schema Location : {schema_path}")
    print(f"Checkpoint      : {checkpoint_path}")
    print(f"Target Table    : {target_table}")

    audit_schema = StructType([
    StructField(
        "source_table",
        StringType(),
        True
    ),
    StructField(
        "target_table",
        StringType(),
        True
    ),
    StructField(
        "start_time",
        TimestampType(),
        True
    ),
    StructField(
        "end_time",
        TimestampType(),
        True
    ),
    StructField(
        "duration_seconds",
        DoubleType(),
        True
    ),
    StructField(
        "status",
        StringType(),
        True
    ),
    StructField(
        "error_message",
        StringType(),
        True
    ),
    StructField(
        "record_count",
        LongType(),
        True
    )
    ])

    try:
            df = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format",source_format)\
                    .option("cloudFiles.schemaLocation",schema_path)\
                        .load(source_path)

            df = (df.withColumn("_ingestion_timestamp",F.current_timestamp())\
                .withColumn("_source_file",F.col("_metadata.file_path"))          
                )
            
            query = df.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation", checkpoint_path)\
                    .trigger(availableNow = True)\
                        .toTable(target_table)
            
            query.awaitTermination()

            end_time = datetime.now()
            duration = (end_time-start_time).total_seconds()

            current_count = spark.sql(
            f"""
            SELECT COUNT(*) AS count
            FROM {target_table}
            """
            ).collect()[0]["count"]
            
            record_count = current_count - previous_count

            if record_count > 0:
                status = "SUCCESS"
            else:
                status = "NO_DATA"

            audit_data = [
            (
                source_table,
                target_table,
                start_time,
                end_time,
                duration,
                status,
                None,
                record_count
            )
            ]

            audit_df = spark.createDataFrame(
            audit_data,
            audit_schema
            )

            audit_df.write.mode("append").saveAsTable(
            "quickcart.bronze.bronze_ingestion_audit"
            )

            print(
            f"{status}: {source_table} | "
            f"Records: {record_count} | "
            f"Duration: {duration:.2f} sec"
            )

            return status

    except Exception as e:
            end_time = datetime.now()

            duration = (
                end_time - start_time
            ).total_seconds()

            error_message = str(e)
            
            audit_data = [
            (
                source_table,
                target_table,
                start_time,
                end_time,
                duration,
                "FAILED",
                error_message,
                0
            )
            ]

            audit_df = spark.createDataFrame(
            audit_data,
            audit_schema
            )

            audit_df.write.mode("append").saveAsTable(
            "quickcart.bronze.bronze_ingestion_audit"
            )

            print(
            f"FAILED: {source_table}"
            )

            print(
            f"Error: {error_message}"
            )

            return "FAILED"
            

In [0]:
ingest_to_bronze("customers")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM quickcart.bronze.customers
""").show()

# TEST

###Create 100 new records

In [0]:
# products_source =  spark.read.parquet(
#     "/Volumes/quickcart/default/source_data/products"
# )

# products_source.count()

In [0]:
# new_products = products_source.limit(50).withColumn("product_id",
#                                                      F.concat(F.lit("NEW_TEST"),
#                                                               F.monotonically_increasing_id().cast("string")))

# #display(new_products)

# new_products.write.mode("append").parquet("/Volumes/quickcart/default/source_data/products")

In [0]:
# products_after_new = spark.read.parquet("/Volumes/quickcart/default/source_data/products")
# products_after_new.count()

In [0]:
# display(
#     dbutils.fs.ls(
#         "/Volumes/quickcart/default/source_data/products"
#     )
# )

###Creating 10 NEW records of Customers for Silver Incremental Testing

In [0]:
# customer_source =  spark.read.parquet(
#     "/Volumes/quickcart/default/source_data/customers"
# )
# #customer_source.count()

# new_customers = customer_source.limit(10).withColumn("customer_id",
#                                                      F.concat(F.lit("NEW_TEST"),
#                                                               F.monotonically_increasing_id().cast("string")))

# display(new_customers)

# new_customers.write.mode("append").parquet("/Volumes/quickcart/default/source_data/customers")



In [0]:
spark.table("quickcart.bronze.customers").agg(
    F.min("updated_at").alias("min_updated_at"),
    F.max("updated_at").alias("max_updated_at")
).show()

In [0]:
spark.table("quickcart.bronze.customers").filter(
    F.col("updated_at").isNull()
).count()

In [0]:
spark.table("quickcart.silver.customers").count()

display(
    spark.table("quickcart.silver.silver_audit")
    .filter(F.col("table_name") == "customers")
    .orderBy(F.col("start_time").desc())
)

In [0]:
%sql
--SELECT MAX(updated_at), MIN(updated_at) FROM quickcart.bronze.customers WHERE customer_id like 'NEW_CUST_%';

--SELECT MAX(updated_at), MIN(updated_at) FROM quickcart.bronze.customers WHERE customer_id like 'INC_CUST_%';